# 01 - Data Cleaning & Feature Engineering M15

**Tujuan:** Membersihkan dan menyiapkan data candle M15 EURUSD untuk digunakan sebagai input model AI prediksi sinyal trading.

**Output akhir:** Dataset bersih dengan label:
- `signal` → arah candle berikutnya: `BUY` / `SELL` / `HOLD`
- `sl` → Stop Loss (berbasis ATR)
- `tp` → Take Profit (berbasis ATR)

**File sumber:** `EURUSD.m_M15_202112160545_202603251830.csv`  
**File output:** `EURUSD_M15_cleaned.csv`

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


## 2. Load Raw Data

In [2]:
RAW_FILE = 'EURUSD.m_M15_202112160545_202603251830.csv'
OUTPUT_FILE = 'EURUSD_M15_cleaned.csv'

df = pd.read_csv(RAW_FILE, sep='\t')

print(f'Shape awal: {df.shape}')
print(f'Kolom: {df.columns.tolist()}')
df.head()

Shape awal: (100267, 9)
Kolom: ['<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>', '<CLOSE>', '<TICKVOL>', '<VOL>', '<SPREAD>']


,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<TICKVOL>,<VOL>,<SPREAD>
0,2021.12.16,05:45:00,1.12871,1.12878,1.12854,1.12856,88,0,2
1,2021.12.16,06:00:00,1.12857,1.12866,1.12849,1.12854,109,0,2
2,2021.12.16,06:15:00,1.12853,1.12869,1.12835,1.12849,118,0,2
3,2021.12.16,06:30:00,1.12848,1.12890,1.12846,1.12887,155,0,2
4,2021.12.16,06:45:00,1.12887,1.12894,1.12861,1.12863,169,0,2


## 3. Cleaning - Rename & Parse Datetime

In [3]:
# Rename kolom ke format standar
df.rename(columns={
    '<DATE>': 'date',
    '<TIME>': 'time',
    '<OPEN>': 'open',
    '<HIGH>': 'high',
    '<LOW>': 'low',
    '<CLOSE>': 'close',
    '<TICKVOL>': 'tick_vol',
    '<VOL>': 'vol',
    '<SPREAD>': 'spread'
}, inplace=True)

# Gabungkan date + time jadi satu kolom datetime
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='%Y.%m.%d %H:%M:%S')
df.set_index('datetime', inplace=True)
df.drop(columns=['date', 'time', 'vol'], inplace=True)

print(f'Range data: {df.index.min()} → {df.index.max()}')
print(f'Total candle: {len(df):,}')
df.head()

Range data: 2021-12-16 05:45:00 → 2026-03-25 18:30:00
Total candle: 100,267


,open,high,low,close,tick_vol,spread
datetime,,,,,,
2021-12-16 05:45:00,1.12871,1.12878,1.12854,1.12856,88,2
2021-12-16 06:00:00,1.12857,1.12866,1.12849,1.12854,109,2
2021-12-16 06:15:00,1.12853,1.12869,1.12835,1.12849,118,2
2021-12-16 06:30:00,1.12848,1.12890,1.12846,1.12887,155,2
2021-12-16 06:45:00,1.12887,1.12894,1.12861,1.12863,169,2


## 4. Cleaning - Cek Missing Value & Duplikat

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print(f'\n=== Duplikat ===')
print(f'Jumlah duplikat: {df.duplicated().sum()}')

# Hapus duplikat jika ada
df = df[~df.duplicated()]

print(f'\n=== Candle tidak valid (high < low) ===')
invalid = df[df['high'] < df['low']]
print(f'Jumlah: {len(invalid)}')
df = df[df['high'] >= df['low']]

print(f'\nShape setelah cleaning: {df.shape}')

## 5. Cek Gap Waktu (Missing Candle)

In [ ]:
# Hitung gap antara candle
df_sorted = df.sort_index()
time_diff = df_sorted.index.to_series().diff().dropna()
expected = pd.Timedelta(minutes=15)

gaps = time_diff[time_diff > expected]
print(f'Total gap (lebih dari 15 menit): {len(gaps)}')
print(f'Gap terbesar: {gaps.max()}')
print(f'\n10 gap terbesar:')
print(gaps.sort_values(ascending=False).head(10))

## 6. Feature Engineering - Indikator Teknikal

In [ ]:
# --- EMA (Trend Filter) ---
df['ema_9']   = df['close'].ewm(span=9, adjust=False).mean()
df['ema_21']  = df['close'].ewm(span=21, adjust=False).mean()
df['ema_50']  = df['close'].ewm(span=50, adjust=False).mean()
df['ema_200'] = df['close'].ewm(span=200, adjust=False).mean()

# --- RSI (14) ---
delta = df['close'].diff()
gain  = delta.clip(lower=0)
loss  = -delta.clip(upper=0)
avg_gain = gain.ewm(span=14, adjust=False).mean()
avg_loss = loss.ewm(span=14, adjust=False).mean()
rs = avg_gain / avg_loss
df['rsi_14'] = 100 - (100 / (1 + rs))

# --- MACD (12, 26, 9) ---
ema12 = df['close'].ewm(span=12, adjust=False).mean()
ema26 = df['close'].ewm(span=26, adjust=False).mean()
df['macd_line']   = ema12 - ema26
df['macd_signal'] = df['macd_line'].ewm(span=9, adjust=False).mean()
df['macd_hist']   = df['macd_line'] - df['macd_signal']

# --- Bollinger Bands (20, 2) ---
bb_mid = df['close'].rolling(window=20).mean()
bb_std = df['close'].rolling(window=20).std()
df['bb_upper'] = bb_mid + (2 * bb_std)
df['bb_mid']   = bb_mid
df['bb_lower'] = bb_mid - (2 * bb_std)

# --- ATR (14) - untuk SL/TP ---
high_low   = df['high'] - df['low']
high_close = (df['high'] - df['close'].shift()).abs()
low_close  = (df['low']  - df['close'].shift()).abs()
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['atr_14'] = tr.ewm(span=14, adjust=False).mean()

print('Indikator berhasil dihitung.')
print(df[['close', 'ema_9', 'ema_21', 'rsi_14', 'macd_line', 'atr_14']].tail())

## 7. Feature Engineering - Candle Pattern

In [ ]:
# Body & shadow candle
df['body']         = (df['close'] - df['open']).abs()
df['upper_shadow']  = df['high'] - df[['open', 'close']].max(axis=1)
df['lower_shadow']  = df[['open', 'close']].min(axis=1) - df['low']
df['candle_dir']    = np.where(df['close'] >= df['open'], 1, -1)  # 1=bullish, -1=bearish

# Return candle (perubahan harga)
df['return']        = df['close'].pct_change()
df['return_3']      = df['close'].pct_change(3)   # return 3 candle terakhir
df['return_5']      = df['close'].pct_change(5)

print('Candle features ditambahkan.')
df[['open','high','low','close','body','upper_shadow','lower_shadow','candle_dir']].tail()

## 8. Labeling - Signal BUY / SELL / HOLD

In [ ]:
# --- Parameter ---
SL_MULTIPLIER  = 1.0   # SL = 1x ATR dari entry
TP_MULTIPLIER  = 2.0   # TP = 2x ATR dari entry (RR 1:2)
LOOKAHEAD      = 4     # Lihat 4 candle ke depan (1 jam)

signals = []
sl_values = []
tp_values = []

close_arr = df['close'].values
atr_arr   = df['atr_14'].values

for i in range(len(df)):
    if i + LOOKAHEAD >= len(df):
        signals.append('HOLD')
        sl_values.append(np.nan)
        tp_values.append(np.nan)
        continue

    entry = close_arr[i]
    atr   = atr_arr[i]

    if np.isnan(atr) or atr == 0:
        signals.append('HOLD')
        sl_values.append(np.nan)
        tp_values.append(np.nan)
        continue

    # Harga future dalam window LOOKAHEAD
    future_prices = close_arr[i+1 : i+1+LOOKAHEAD]
    max_future = np.max(future_prices)
    min_future = np.min(future_prices)

    sl_buy  = entry - (SL_MULTIPLIER * atr)
    tp_buy  = entry + (TP_MULTIPLIER * atr)
    sl_sell = entry + (SL_MULTIPLIER * atr)
    tp_sell = entry - (TP_MULTIPLIER * atr)

    hit_tp_buy  = max_future >= tp_buy
    hit_sl_buy  = min_future <= sl_buy
    hit_tp_sell = min_future <= tp_sell
    hit_sl_sell = max_future >= sl_sell

    if hit_tp_buy and not hit_sl_buy:
        signals.append('BUY')
        sl_values.append(round(sl_buy, 5))
        tp_values.append(round(tp_buy, 5))
    elif hit_tp_sell and not hit_sl_sell:
        signals.append('SELL')
        sl_values.append(round(sl_sell, 5))
        tp_values.append(round(tp_sell, 5))
    else:
        signals.append('HOLD')
        sl_values.append(np.nan)
        tp_values.append(np.nan)

df['signal'] = signals
df['sl']     = sl_values
df['tp']     = tp_values

print('Distribusi signal:')
print(df['signal'].value_counts())
print(f'\nPersentase:')
print(df['signal'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

## 9. Visualisasi Distribusi Signal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Pie chart
counts = df['signal'].value_counts()
colors = {'BUY': '#2ecc71', 'SELL': '#e74c3c', 'HOLD': '#95a5a6'}
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=[colors[k] for k in counts.index])
axes[0].set_title('Distribusi Signal')

# Sample price + signal (200 candle terakhir)
sample = df.tail(200).copy()
axes[1].plot(sample.index, sample['close'], color='gray', linewidth=0.8, label='Close')
buy_pts  = sample[sample['signal'] == 'BUY']
sell_pts = sample[sample['signal'] == 'SELL']
axes[1].scatter(buy_pts.index,  buy_pts['close'],  color='green', marker='^', s=40, label='BUY', zorder=5)
axes[1].scatter(sell_pts.index, sell_pts['close'], color='red',   marker='v', s=40, label='SELL', zorder=5)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].set_title('Signal pada 200 Candle Terakhir')

plt.tight_layout()
plt.show()

## 10. Final Check & Export

In [ ]:
# Drop baris dengan NaN di indikator utama (warmup period)
required_cols = ['ema_200', 'rsi_14', 'macd_line', 'bb_mid', 'atr_14']
df_clean = df.dropna(subset=required_cols).copy()

print(f'Shape sebelum drop NaN: {df.shape}')
print(f'Shape setelah drop NaN: {df_clean.shape}')
print(f'Candle dihapus (warmup): {len(df) - len(df_clean)}')
print(f'\nDistribusi signal final:')
print(df_clean['signal'].value_counts())

# Preview kolom final
print(f'\nKolom output ({len(df_clean.columns)}):')
print(df_clean.columns.tolist())

In [ ]:
# Export ke CSV
df_clean.to_csv(OUTPUT_FILE)
print(f'File disimpan: {OUTPUT_FILE}')
print(f'Total baris: {len(df_clean):,}')
df_clean.tail()